# iPhone Resale Price Analytics | EDA + Machine Learning

**Portfolio-ready Google Colab notebook** · Data audit → EDA → leakage-aware ML → evaluation → exports.

**研究問題：** 二手 iPhone 的報價如何隨型號、容量、狀態、平台與時間改變？我們能否預測同一幣別市場的報價？

**Important caveats:** Dataset consists of **partner offer / buyback quotes**, not completed transaction prices. Currency is **not converted**; price scales differ by currency. Verified means *source currency label confirmed*, not that every field is error-free. Product-month summaries are not independent transactions. This is an exploratory portfolio project, not a deployable valuation service.

**Instructions:** Colab → upload `iPhone_Resale_Final_Primary_Verified.csv` when prompted → Runtime → Run all. Output artifacts appear in `iphone_resale_outputs/` and a ZIP download is offered at the end.

## 1. Environment & load data
No external exchange-rate API needed: each model is trained and evaluated **within one currency**, avoiding misleading cross-currency comparisons.

In [ ]:
import os, re, json, warnings, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
warnings.filterwarnings('ignore', category=FutureWarning)
SEED=42
OUT=Path('iphone_resale_outputs'); OUT.mkdir(exist_ok=True)
filename='iPhone_Resale_Final_Primary_Verified.csv'
if not Path(filename).exists():
    try:
        from google.colab import files
        print('Please upload:',filename)
        files.upload()
    except ImportError:
        raise FileNotFoundError(f'Place {filename} in the notebook working directory.')
raw=pd.read_csv(filename)
print('Loaded:',raw.shape)
display(raw.head(3))

## 2. Data quality audit & safe modeling subset
We do **not** silently repair ambiguous capacity strings such as `15512GB`, nor infer missing currencies. Such rows are excluded from modeling and exported for review. Supported capacities are 16, 32, 64, 128, 256, 512 GB, 1 or 2 TB. Date anomalies are **flagged**, not automatically rewritten. Raw dataset remains untouched.

In [ ]:
df=raw.copy()
required=['Partner','Model','Capacity','Condition','Quote_Month','Price_Original','Currency','Currency_Verified']
assert not (missing:=[c for c in required if c not in df.columns]), f'Missing columns: {missing}'
df['Quote_Month']=pd.to_datetime(df['Quote_Month'],errors='coerce')
df['Price_Original']=pd.to_numeric(df['Price_Original'],errors='coerce')
for col in ['Partner','Model','Capacity','Condition','Currency']:
    df[col]=df[col].astype('string').str.strip()
valid_capacities={'16GB','32GB','64GB','128GB','256GB','512GB','1TB','2TB'}
df['Capacity_Canonical']=df['Capacity'].str.upper().str.replace(' ','',regex=False)
verified=df['Currency_Verified'].astype(str).str.lower().isin(['true','1','yes'])
reasons=pd.DataFrame(index=df.index)
reasons['Unverified_Currency']=~verified
reasons['Invalid_Currency']=~df['Currency'].isin(['USD','GBP','AED','SAR']).fillna(False)
reasons['Invalid_Price']=~(df['Price_Original'].notna() & np.isfinite(df['Price_Original']) & (df['Price_Original']>0))
reasons['Invalid_Date']=df['Quote_Month'].isna()
reasons['Ambiguous_Capacity']=~df['Capacity_Canonical'].isin(valid_capacities).fillna(False)
reasons['Missing_Identity']=df[['Partner','Model','Condition']].isna().any(axis=1) | df[['Partner','Model','Condition']].eq('').any(axis=1)
df['Exclude_From_Model']=reasons.any(axis=1)
df['Quality_Review_Reason']=reasons.apply(lambda r:'; '.join(r.index[r.to_numpy()]),axis=1)
review=df[df['Exclude_From_Model']].copy()
clean=df[~df['Exclude_From_Model']].copy()
clean['Year']=clean['Quote_Month'].dt.year.astype(int)
clean['Month_Number']=clean['Quote_Month'].dt.month.astype(int)
clean['Month_Index']=clean['Year']*12+clean['Month_Number']
clean['Capacity_GB']=clean['Capacity_Canonical'].map(lambda x: int(x[:-2])*1024 if x.endswith('TB') else int(x[:-2]))
print('Original:',len(df),'Modeling eligible:',len(clean),'Needs review:',len(review))
print('Review reason counts:');display(reasons.sum().sort_values(ascending=False).to_frame('Rows'))
print('Currency / platform coverage:');display(pd.crosstab(clean['Partner'],clean['Currency']))
print('Month counts:');display(clean['Quote_Month'].dt.to_period('M').value_counts().sort_index().to_frame('Rows'))
print('Potentially anomalous earliest dates:');display(clean['Quote_Month'].dt.to_period('M').value_counts().sort_index().head(5))
review.to_csv(OUT/'excluded_for_modeling_review.csv',index=False)
clean.to_csv(OUT/'modeling_eligible_native_currency.csv',index=False)
assert len(clean)+len(review)==len(df)

## 3. EDA — source mix and price distributions
**Interpretation:** Prices are in native currency. Graphs are **split by currency**; do not compare absolute AED vs GBP vs USD vs SAR values. Platform quote mixes differ in product and condition, so raw platform averages do not establish which platform offers the best deal.

In [ ]:
counts=clean.groupby(['Currency','Partner']).size().sort_values(ascending=False)
display(counts.rename('Quotes').reset_index())
fig,axs=plt.subplots(1,2,figsize=(14,5))
counts.sort_values().plot.barh(ax=axs[0],title='Quote counts by currency / partner')
axs[0].set_xlabel('Number of rows')
for cur,g in clean.groupby('Currency'):
    axs[1].hist(g['Price_Original'],bins=30,alpha=.4,label=f'{cur} (n={len(g)})')
axs[1].set_title('Native-currency price distributions (NOT comparable)')
axs[1].set_xlabel('Original currency units');axs[1].legend()
plt.tight_layout();plt.savefig(OUT/'01_source_and_prices.png',dpi=160,bbox_inches='tight');plt.show()
display(clean.groupby('Currency')['Price_Original'].agg(['count','median','min','max']).round(2))

## 4. EDA — iPhone models, capacities, condition
For interpretable charts, use **USD (SellCell)** where available, otherwise the largest single-currency subset. Descriptive group medians can be confounded by product mix, dates and source methodology.

In [ ]:
chosen='USD' if (clean['Currency']=='USD').sum()>=100 else clean['Currency'].value_counts().idxmax()
s=clean[clean['Currency']==chosen].copy()
model_stats=s.groupby('Model')['Price_Original'].agg(['size','median']).query('size>=5').sort_values('size',ascending=False).head(15)
fig,axs=plt.subplots(1,3,figsize=(17,5))
model_stats.sort_values('median')['median'].plot.barh(ax=axs[0],title=f'Median by model ({chosen})')
s.groupby('Capacity_Canonical')['Price_Original'].median().sort_values().plot.bar(ax=axs[1],title=f'Median by capacity ({chosen})')
s.groupby('Condition')['Price_Original'].median().sort_values().plot.barh(ax=axs[2],title=f'Median by condition ({chosen})')
for a in axs:a.set_ylabel(f'Price ({chosen})')
plt.tight_layout();plt.savefig(OUT/'02_model_capacity_condition.png',dpi=160,bbox_inches='tight');plt.show()
print('Caution: Different models and conditions may be represented in each capacity bin.')

## 5. EDA — monthly trends
A month-to-month median can move because the **product mix changes**, even if no individual phone depreciates. The second panel tracks repeated product identities and computes paired month-to-month percentage changes only when both months are observed. Treat even these as quote changes, not realized resale depreciation.

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(14,5))
for cur,g in clean.groupby('Currency'):
    monthly=g.groupby('Quote_Month')['Price_Original'].median().sort_index()
    axs[0].plot(monthly.index,monthly.values,marker='o',label=cur)
axs[0].set_title('Monthly median quotes (separate currency scales)')
axs[0].set_ylabel('Native currency units');axs[0].legend();axs[0].tick_params(axis='x',rotation=40)
identity=['Partner','Currency','Model','Capacity_Canonical','Condition']
agg=clean.groupby(identity+['Quote_Month'],dropna=False)['Price_Original'].median().reset_index()
agg=agg.sort_values(identity+['Quote_Month'])
agg['Previous_Price']=agg.groupby(identity,dropna=False)['Price_Original'].shift()
agg['Previous_Date']=agg.groupby(identity,dropna=False)['Quote_Month'].shift()
agg['Month_Gap']=(agg['Quote_Month'].dt.year-agg['Previous_Date'].dt.year)*12+(agg['Quote_Month'].dt.month-agg['Previous_Date'].dt.month)
paired=agg[(agg['Month_Gap']==1)&(agg['Previous_Price']>0)].copy()
paired['Change_Pct']=100*(paired['Price_Original']/paired['Previous_Price']-1)
paired=paired[np.isfinite(paired['Change_Pct'])]
if len(paired):
    paired.groupby('Currency')['Change_Pct'].median().sort_values().plot.bar(ax=axs[1],title='Median within-product monthly change')
else:axs[1].text(.5,.5,'No adjacent-month product pairs',ha='center',transform=axs[1].transAxes)
axs[1].set_ylabel('Median change (%)');plt.tight_layout()
plt.savefig(OUT/'03_monthly_trends.png',dpi=160,bbox_inches='tight');plt.show()
print('Adjacent-month pairs:',len(paired));display(paired.groupby('Currency')['Change_Pct'].agg(['count','median']).round(2) if len(paired) else pd.DataFrame())

## 6. ML design — native-currency, leakage-aware evaluation

- **Separate model for each currency.** No FX conversion or cross-currency target mixing.
- **Target:** `Price_Original` (partner quote, not transaction price).
- **Features:** `Partner`, `Model`, `Capacity_GB`, `Condition`, `Month_Index`.
- **Group holdout:** all observations of a given `(Partner, Model, Capacity, Condition)` identity stay on one side of the split. This tests unseen product-condition combinations and reduces repeated-quote leakage.
- **Baseline:** median price predictor; **candidate:** Random Forest. Metrics: MAE, RMSE, R², WAPE (with caution).
- **No price-derived columns** (`Price_Min`, `Price_Max`, recorded count) used as predictors. No `Source_Row` or `Zension_Grade` to avoid potential leakage / grading incompatibility.
- Product overlap and chronological generalization remain limitations; group holdout is not a true future-month forecast.

In [ ]:
features=['Partner','Model','Capacity_GB','Condition','Month_Index']
cat=['Partner','Model','Condition'];num=['Capacity_GB','Month_Index']
model_results=[]; predictions=[]; fitted_models={}
for cur,g in clean.groupby('Currency'):
    g=g.dropna(subset=features+['Price_Original']).copy()
    group_id=g[['Partner','Model','Capacity_Canonical','Condition']].astype(str).agg('||'.join,axis=1)
    if len(g)<80 or group_id.nunique()<12:
        print('SKIP',cur,'insufficient rows or distinct groups');continue
    splitter=GroupShuffleSplit(n_splits=1,test_size=.2,random_state=SEED)
    tr,te=next(splitter.split(g,groups=group_id))
    train,test=g.iloc[tr],g.iloc[te]
    Xtr,Xte=train[features],test[features];ytr,yte=train['Price_Original'],test['Price_Original']
    prep=ColumnTransformer([
        ('categorical',Pipeline([('impute',SimpleImputer(strategy='most_frequent')),('encode',OneHotEncoder(handle_unknown='ignore'))]),cat),
        ('numeric',SimpleImputer(strategy='median'),num)
    ])
    estimators={'Median_Baseline':DummyRegressor(strategy='median'),
                'Random_Forest':RandomForestRegressor(n_estimators=200,min_samples_leaf=2,random_state=SEED,n_jobs=-1)}
    for name,est in estimators.items():
        pipe=Pipeline([('prep',prep),('model',est)])
        pipe.fit(Xtr,ytr)
        yp=pipe.predict(Xte)
        mae=mean_absolute_error(yte,yp);rmse=np.sqrt(mean_squared_error(yte,yp))
        wape=100*np.abs(yte-yp).sum()/np.abs(yte).sum() if np.abs(yte).sum()>0 else np.nan
        model_results.append({'Currency':cur,'Model':name,'Train_Rows':len(tr),'Test_Rows':len(te),
                              'Train_Groups':group_id.iloc[tr].nunique(),'Test_Groups':group_id.iloc[te].nunique(),
                              'MAE':mae,'RMSE':rmse,'R2':r2_score(yte,yp),'WAPE_Pct':wape})
        if name=='Random_Forest':
            fitted_models[cur]=pipe
            part=test[['Partner','Model','Capacity_Canonical','Condition','Quote_Month']].copy()
            part['Currency']=cur;part['Actual']=yte.to_numpy();part['Predicted']=yp
            part['Absolute_Error']=np.abs(part['Actual']-part['Predicted']);predictions.append(part)
results=pd.DataFrame(model_results)
assert len(results)>0,'No eligible currency groups for training'
print('GROUP HOLDOUT RESULTS (test data only)')
display(results.round(3).sort_values(['Currency','Model']))
results.to_csv(OUT/'ml_model_comparison.csv',index=False)
pred_df=pd.concat(predictions,ignore_index=True)
pred_df.to_csv(OUT/'ml_test_predictions.csv',index=False)

## 7. Model evaluation — actual vs predicted and baseline lift
Metrics **cannot** be compared directly across currencies. Check if the forest beats the baseline **within each currency**. High R² alone does not prove causation, fairness, or future performance.

In [ ]:
fig,axs=plt.subplots(1,len(fitted_models),figsize=(5*len(fitted_models),4),squeeze=False)
for ax,(cur,model) in zip(axs[0],fitted_models.items()):
    p=pred_df[pred_df['Currency']==cur]
    ax.scatter(p['Actual'],p['Predicted'],alpha=.3,s=12)
    low=min(p[['Actual','Predicted']].min());high=max(p[['Actual','Predicted']].max())
    ax.plot([low,high],[low,high],linestyle='--',color='black')
    ax.set(title=f'{cur}: actual vs predicted',xlabel=f'Actual ({cur})',ylabel=f'Predicted ({cur})')
plt.tight_layout();plt.savefig(OUT/'04_ml_actual_vs_predicted.png',dpi=160,bbox_inches='tight');plt.show()
pivot=results.pivot(index='Currency',columns='Model',values='MAE')
pivot['MAE_Improvement_Pct']=100*(pivot['Median_Baseline']-pivot['Random_Forest'])/pivot['Median_Baseline']
print('Positive improvement means forest beats baseline:');display(pivot.round(2))
pivot.to_csv(OUT/'ml_baseline_lift.csv')

## 8. Feature importance (descriptive, not causal)
Forest impurity importance can be biased. Instead, use **permutation importance on held-out data**, grouped at the original feature level. It measures how much MAE worsens when a feature is shuffled; correlated predictors complicate interpretation.

In [ ]:
from sklearn.inspection import permutation_importance
importance_rows=[]
for cur,pipe in fitted_models.items():
    g=clean[clean['Currency']==cur].dropna(subset=features+['Price_Original']).copy()
    gids=g[['Partner','Model','Capacity_Canonical','Condition']].astype(str).agg('||'.join,axis=1)
    _,te=next(GroupShuffleSplit(n_splits=1,test_size=.2,random_state=SEED).split(g,groups=gids))
    tst=g.iloc[te]
    perm=permutation_importance(pipe,tst[features],tst['Price_Original'],n_repeats=5,
                                random_state=SEED,scoring='neg_mean_absolute_error',n_jobs=1)
    for f,v in zip(features,perm.importances_mean):importance_rows.append({'Currency':cur,'Feature':f,'MAE_Increase_When_Shuffled':v})
imp=pd.DataFrame(importance_rows)
display(imp.sort_values(['Currency','MAE_Increase_When_Shuffled'],ascending=[True,False]).round(2))
imp.to_csv(OUT/'ml_permutation_importance.csv',index=False)
fig,ax=plt.subplots(figsize=(9,5))
imp.pivot(index='Feature',columns='Currency',values='MAE_Increase_When_Shuffled').plot.barh(ax=ax)
ax.set_xlabel('MAE increase (each currency in own units; do not compare magnitudes)')
plt.tight_layout();plt.savefig(OUT/'05_feature_importance.png',dpi=160,bbox_inches='tight');plt.show()

## 9. Export portfolio deliverables
Files include cleaned modeling subset, excluded-row audit, EDA charts, model metrics, test predictions, feature importance, and README. The original source is **not modified**. Download the ZIP for GitHub; upload only data you have permission to redistribute.

In [ ]:
summary={
 'input_rows':int(len(raw)),'modeling_eligible_rows':int(len(clean)),
 'excluded_for_review_rows':int(len(review)),
 'currencies':clean['Currency'].value_counts().to_dict(),
 'split':'GroupShuffleSplit, 20% holdout, seed 42, groups Partner+Model+Capacity+Condition',
 'target':'Price_Original (native-currency partner quote)',
 'note':'No cross-currency pooled model. No FX conversion. Group split is not future-time validation.'
}
(OUT/'run_summary.json').write_text(json.dumps(summary,indent=2,ensure_ascii=False,default=str),encoding='utf-8')
readme="""# iPhone Resale Price Analytics

## Objective
Explore partner buyback quotes and predict quoted iPhone resale prices within each currency.

## Data
Input: `iPhone_Resale_Final_Primary_Verified.csv` (precleaned source). Quote prices are not completed transactions. Multiple native currencies (USD, GBP, AED, SAR); **not FX converted**. Currency verified does not imply all other fields are verified.

## Reproducibility
Open `iPhone_Resale_EDA_ML_Portfolio.ipynb` in Google Colab, upload the input CSV and run all cells. Python: pandas, numpy, matplotlib, scikit-learn.

## Cleaning & exclusions
Review anomalous capacities, missing identities, missing dates, nonpositive prices and unverified currencies. Export `excluded_for_modeling_review.csv`. Source file remains unchanged. Check unusually early dates before interpreting trends.

## EDA
Source coverage, price distributions, model/capacity/condition, monthly medians, paired adjacent-month changes. Descriptive trends may be confounded by changing quote mix.

## ML design
Separate models by currency; GroupShuffleSplit by Partner+Model+Capacity+Condition; median baseline vs Random Forest. Report holdout MAE, RMSE, R², WAPE. Do not interpret this split as forward-in-time forecasting. Price range, row IDs, and other target-derived fields are excluded from predictors.

## Limitations
Nonrandom quote sampling; heterogeneous partner grading; repeated product-month aggregates; uncertain timestamps; excluded unverified quotes; potential remaining source errors; small or sparse subgroup sizes. Correlation does not establish causation. Model performance is not a guarantee for unseen time periods.

## Next improvements
Verify unusual capacities/dates against original Excel; check licensing; build forward-time validation; consider historical FX if cross-market price comparison is necessary; analyze grade mapping and pricing consistency.
"""
(OUT/'README.md').write_text(readme,encoding='utf-8')
zipname='iPhone_Resale_Portfolio_Results.zip'
with zipfile.ZipFile(zipname,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.iterdir()):
        if p.is_file():z.write(p,arcname=p.name)
print('Saved:',zipname,'with',len(list(OUT.iterdir())),'files')
try:
    from google.colab import files
    files.download(zipname)
except ImportError:
    print('ZIP available in current working directory.')

## Portfolio discussion prompts

1. Why is it incorrect to compare raw AED and GBP quotes?  
2. Why might median prices change across months even if every model depreciates?  
3. How does grouping by product-condition identity reduce train/test leakage?  
4. When does a Random Forest outperform the median baseline? For which currencies?  
5. Why do buyback offers differ from consumer resale transaction prices?  
6. What data would be needed for true future-month price forecasting?

**Suggested GitHub title:** `iphone-resale-price-analytics`  
**Suggested subtitle:** *Multi-market iPhone buyback quote EDA and leakage-aware price prediction.*